# Langchain

LangChain is an open-source framework for building applications powered by large language models (LLMs). It provides tools, abstractions, and composable "chains" to connect LLMs with data sources, memory, tools, and external APIs—commonly used for chatbots, RAG (retrieval-augmented generation), and agents. It has Python and JavaScript/TypeScript libraries.

LangChain
What It Is
LangChain is an open-source framework for building applications powered by large language models (LLMs). It gives you composable building blocks (chains, prompts, models, memory, tools, agents) so you can wire LLMs into real software instead of calling an API in one shot.
Core idea: an LLM call + context + tools + memory = an application.
There are two main Python packages:
- langchain — the core orchestration
- langchain-community / langchain-openai etc. — integrations
- langgraph — for stateful, multi-step agent workflows (the modern way to build agents)
- langserve — deploy chains as APIs
- LangSmith — observability/debugging platform
Why Use It
- Composability: swap models, prompts, vector stores without rewriting everything.
- Retrieval (RAG): easy glue between your data and the LLM via vector DBs.
- Agents & tools: let the LLM decide to call APIs, search, run code.
- Memory: maintain conversation state across turns.
- Observability: LangSmith traces every step for debugging.
- Ecosystem: hundreds of integrations (OpenAI, Anthropic, Pinecone, etc.).
How To Use It (basic example)
pip install langchain langchain-openai langchain-community
A simple chain (modern LCEL style):
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o")
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "{input}")
])
chain = prompt | llm
print(chain.invoke({"input": "What is LangChain?"}))
RAG example with a vector store:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

docs = ["LangChain helps build LLM apps.", "It supports RAG and agents."]
vectorstore = FAISS.from_texts(docs, OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
    "Answer using context:\n{context}\n\nQuestion: {question}"
)
llm = ChatOpenAI()
rag_chain = (
    {"context": retriever, "question": lambda x: x["question"]}
    | prompt | llm | StrOutputParser()
)
print(rag_chain.invoke({"question": "What does LangChain support?"}))
Agent with tools (LangGraph-style):
from langchain_openai import ChatOpenAI
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"Sunny in {city}"

llm = ChatOpenAI()
llm_with_tools = llm.bind_tools([get_weather])
Common Use Cases
- RAG / Q&A bots: answer questions over your docs, PDFs, knowledge base.
- Chatbots with memory: customer support, assistants.
- Agents: automate tasks (booking, data lookup, API orchestration).
- Document summarization & extraction.
- Structured output: turn text into JSON for pipelines.
- Multi-agent systems via LangGraph (planning, reflection, workflows).
When NOT To Use It
- For a single simple API call, raw SDK (openai/anthropic) is simpler.
- Adds abstraction/dependencies; for tiny scripts, it may be overkill.
Key Takeaway
LangChain = the standard toolkit for putting LLMs into production apps, especially when you need retrieval, tools, memory, or multi-step agent logic. In 2026, LangGraph is the recommended approach for anything agentic/stateful.

Bare Structure of a LangChain App
Below is the minimal skeleton — pseudocode showing only the shape of every example we built.
#### 1. MODEL  ── the LLM endpoint (local or cloud)
llm = ChatModel(provider="ollama", model="<model-name>")
####        ↑ connects to a server that turns text → completion

#### 2. PROMPT ── a reusable template with {variables}
prompt = Template(
    "Do <task> using {input}."   # placeholders filled at runtime
)
####        ↑ just a string-builder; no model call yet

#### 3. CHAIN  ── wire prompt → model (→ parser)
chain = prompt | llm | parser
#####        ↑ "|" means: output of left becomes input of right

#### 4. RUN    ── supply variables, get a result
output = chain.invoke({"input": "..."})
####       ↑ actually calls the model and returns the answer
What each piece is responsible for
Piece	Role	Changes between examples?
llm	Talks to the model	Yes — ChatOpenAI vs ChatOllama
prompt	Shapes the input text	Yes — content & variables
chain	Connects the steps	No — always prompt | llm
invoke	Executes with data	No — always passes a dict
The one invariant
prompt → model → (parser) → invoke(variables)
Everything else (roles, parsers, model choice) is decoration on top of this core.

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="mannix/llama3.1-8b-abliterated:q4_k_m")

prompt = ChatPromptTemplate.from_template(
    "Explain the difference between a mutex and a semaphore in OS concurrency. "
    "Include a short code-style analogy. Be precise and technical.\n\nContext: {level}"
)

chain = prompt | llm | StrOutputParser()

print(chain.invoke({"level": "for a senior backend engineer"}))
This adds StrOutputParser() so you get clean text (not an AIMessage object), and uses a technical prompt. Swap the template for anything — e.g.:
# Structured JSON output example
from langchain_core.output_parsers import JsonOutputParser

prompt = ChatPromptTemplate.from_template(
    "Return JSON only: analyze this error and give a fix.\nError: {error}"
)
chain = prompt | llm | JsonOutputParser()
print(chain.invoke({"error": "deadlock on double lock"}))
Both run against your already-running Ollama. Change {level}/{error} at call time.

Here's what we built, step by step:
1. Created the model — ChatOllama(model="mannix/llama3.1-8b-abliterated:q4_k_m") connects to your local Ollama server on port 11434.
2. Built a prompt — ChatPromptTemplate.from_template(...) with {placeholders} that get filled at runtime (e.g. {level}, {error}). Earlier we also used from_messages([...]) to separate system/user/assistant roles.
3. Composed a chain — prompt | llm pipes the formatted prompt into the model. The | operator is LangChain's way of chaining steps.
4. Added an output parser (technical version) — | StrOutputParser() extracts clean text, | JsonOutputParser() forces structured JSON instead of an AIMessage object.
5. Ran it — chain.invoke({"level": "..."}) passes the variables and returns the model's response.
The pattern throughout: prompt → model → (parser) → invoke(variables). Only the model wrapper, prompt content, and parser changed between examples — the composition stayed identical.

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOllama(model="mannix/llama3.1-8b-abliterated:q4_k_m")

# All three roles: system (rules), user (input), assistant (prior reply / history)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a pirate assistant. Always reply in pirate speech."),
    ("user",   "Where be the treasure?"),
    ("assistant", "Arr! The treasure be buried on Skull Island, matey!"),  # prior turn
    ("user",   "{question}"),  # current input
])

chain = prompt | llm

result = chain.invoke({"question": "What should I do next?"})
print(result.content)

Shiver me timbers! Ye gotta find yerself a map, savvy? That scurvy dog, Barnaby Blackbeak, be hidin' it somewhere. I reckon ye can find him at the local tavern, The Black Spot. Just watch yer back, or ye might be walkin' the plank!


Shiver me timbers! Ye gotta find yerself a map, savvy? That scurvy dog, Barnaby Blackbeak, be hidin' it somewhere. I reckon ye can find him at the local tavern, The Black Spot. Just watch yer back, or ye might be walkin' the plank!

In [2]:

result = chain.invoke({"question": "Where are we and who are we?"})
print(result.content)

Shiver me timbers! I be thinkin' ye must be talkin' to yerself, matey. Ye be aboard the Black Dragon, the fastest ship on the seven seas, captained by the infamous Captain Blackbeak hisself! As fer where we be, I reckon we be sailin' through treacherous waters, avoidin' the authorities and plunderin' all the booty we can find!


In [4]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="mannix/llama3.1-8b-abliterated:q4_k_m")

prompt = ChatPromptTemplate.from_template(
    "Explain the difference between a mutex and a semaphore in OS concurrency. "
    "Include a short code-style analogy. Be precise and technical.\n\nContext: {level}"
)

chain = prompt | llm | StrOutputParser()

print(chain.invoke({"level": "for a senior backend engineer"}))

A mutex (short for "mutual exclusion") is a synchronization primitive used to control access to shared resources in concurrent systems, ensuring that only one thread can access the resource at any given time. It provides exclusive access to a particular section of code or data, preventing other threads from modifying it simultaneously.

On the other hand, a semaphore is also a synchronization primitive, but its primary purpose is to limit access to a certain number of resources or slots. It allows a specific number of threads to access the resource and prevents any additional threads from doing so once the limit has been reached.

In terms of analogy:

Mutex (Lock/Unlock):
```c
// Imagine a single elevator in a building
mutex = 1; // only one person can be in the elevator at a time

void enter_elevator() {
  if (!mutex) { // check if someone is already in the elevator
    mutex = 1; // lock the elevator for this thread
    // do some work
    mutex = 0; // unlock the elevator, allow ot

A mutex (short for "mutual exclusion") is a synchronization primitive used to control access to shared resources in concurrent systems, ensuring that only one thread can access the resource at any given time. It provides exclusive access to a particular section of code or data, preventing other threads from modifying it simultaneously.

On the other hand, a semaphore is also a synchronization primitive, but its primary purpose is to limit access to a certain number of resources or slots. It allows a specific number of threads to access the resource and prevents any additional threads from doing so once the limit has been reached.

In terms of analogy:

Mutex (Lock/Unlock):
```c
// Imagine a single elevator in a building
mutex = 1; // only one person can be in the elevator at a time

void enter_elevator() {
  if (!mutex) { // check if someone is already in the elevator
    mutex = 1; // lock the elevator for this thread
    // do some work
    mutex = 0; // unlock the elevator, allow others to enter
  }
}
```

Semaphore (Signal/Broadcast):
```c
// Imagine a limited number of parking spots
semaphore = 5; // allow 5 cars to park

void park_car() {
  if (semaphore > 0) { // check if there's an available spot
    semaphore--; // decrement the count, indicating one less spot left
    // park the car
  } else { // broadcast that no spots are available
    printf("No parking spots available");
  }
}
```

In summary:

1. Mutex: Exclusive access to a shared resource (e.g., a single elevator), ensuring only one thread can modify it at a time.
2. Semaphore: Limited access to a certain number of resources or slots (e.g., limited parking spots), allowing a specific number of threads to access the resource and preventing others once the limit has been reached.

Please note that these analogies are simplified for illustration purposes and may not accurately represent all possible use cases in real-world scenarios.

In [6]:
# Structured JSON output example
from langchain_core.output_parsers import JsonOutputParser

prompt = ChatPromptTemplate.from_template(
    "Return JSON only: analyze this error and give a fix.\nError: {error}"
)
chain = prompt | llm | JsonOutputParser()
print(chain.invoke({"error": "deadlock on double lock"}))

{'error': 'deadlock_on_double_lock', 'fix': 'Avoid using nested locks or critical sections, as they can cause deadlocks. Instead, consider redesigning your code to use alternative synchronization mechanisms, such as using a queue for accessing shared resources or implementing a try-lock/timeout approach.'}
